In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

dbutils.widgets.text("silver_catalog", "silver_germantelekom")
dbutils.widgets.text("gold_catalog", "gold_germantelekom")
dbutils.widgets.text("schema", "dev_customer")

silver_catalog = dbutils.widgets.get("silver_catalog")
gold_catalog   = dbutils.widgets.get("gold_catalog")
schema         = dbutils.widgets.get("schema")

silver_tbl  = f"{silver_catalog}.{schema}.customer_data"
summary_tbl = f"{gold_catalog}.{schema}.customer_summary_by_pincode"

def publish(**values):
    """Hand small values to the next tasks of the job (does nothing outside a job)."""
    for k, v in values.items():
        try:
            dbutils.jobs.taskValues.set(key=k, value=v)
        except Exception:
            print(f"Not running in a job, task value '{k}' not set")

print("Silver:", silver_tbl)
print("Gold  :", summary_tbl)

# COMMAND ----------

# 1. Current customers from silver (old SCD2 versions are excluded)
current_df = spark.table(silver_tbl).filter("is_current = true")
current_n = current_df.count()
if current_n == 0:
    raise Exception(f"No current customers in {silver_tbl}. Silver has not loaded any data.")

# COMMAND ----------

# 2. Summary: customers by pincode, product, speed and segment
today = spark.sql("SELECT current_date()").first()[0]

summary = (current_df
    .groupBy("pincode", "installed_product", "speed_mbps", "segment")
    .agg(F.count("*").alias("customer_count"))
    .withColumn("snapshot_date", F.lit(today))
    .select("pincode", "installed_product", "speed_mbps", "segment",
            "customer_count", "snapshot_date")          # same order as the table
    .cache())

summary_rows = summary.count()
gold_total   = summary.agg(F.sum("customer_count")).first()[0] or 0
print(f"current customers={current_n} | summary rows={summary_rows} | sum of customer_count={gold_total}")

# COMMAND ----------

# 3. Checks BEFORE writing
# 3a. every current customer is counted exactly once
if current_n != gold_total:
    raise Exception(f"Gold reconciliation failed: {current_n} current customers != {gold_total} in the summary")

# 3b. the DataFrame has the same columns and order as the table
table_cols = [f.name for f in spark.table(summary_tbl).schema.fields]
assert summary.columns == table_cols, f"Column mismatch. Code: {summary.columns} | Table: {table_cols}"

# COMMAND ----------

# 4. Write: replace only today's snapshot (a rerun on the same day gives the same result)
(summary.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"snapshot_date = '{today}'")
    .saveAsTable(summary_tbl))

# COMMAND ----------

# 5. Check AFTER writing
in_table = (spark.table(summary_tbl).filter(F.col("snapshot_date") == F.lit(today))
            .agg(F.sum("customer_count")).first()[0] or 0)
if in_table != current_n:
    raise Exception(f"Gold post-write check failed: table has {in_table}, expected {current_n}")

# COMMAND ----------

# 6. Hand the count to the final log task
publish(gold_rows=int(summary_rows))
print(f"Gold done: {summary_rows} summary rows written for {today}")